# Module 6 — Local Herculean Data Cleaning and Verification

Place this notebook and `cleaning.py` in the project root, alongside the existing `data/` folder containing `prices.parquet`, `news.parquet`, and `filings.parquet`.

This notebook reads local files only. It does not download data, install packages, or modify the original Parquet files. Install the project's dependencies before running. Use **Restart Kernel and Run All** twice; save the notebook after the second successful run.

All explanations and code comments are in English. Generated cleaned data and verification evidence are saved under `cleaning_outputs/`. Existing README and dependency files are not rewritten. The standalone script implements the same cells and can be run with `python cleaning.py`.

In [1]:
# Check dependencies without installing or downloading anything.
import importlib.util
required_packages = ['pandas', 'numpy', 'pyarrow']
missing = [name for name in required_packages if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError('Missing packages: ' + ', '.join(missing) + '. Install the project requirements in this Python environment first.')

In [ ]:
# Notebook paths; the standalone script accepts command-line paths.
data_dir = None
output_dir = None
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, importlib.metadata
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if hasattr(value, "to_string") else value)

REPO = 'TheFinAI/Herculean'
SOURCE_VERSION = 'Local snapshot; identify exact files by SHA256. Upstream revision not assumed.'
MAG7 = ['AAPL','AMZN','GOOGL','META','MSFT','NVDA','TSLA']
TABLES = ['prices','news','filings']
# The notebook uses the current project directory; the script supplies explicit paths.
PROJECT_DIR = Path.cwd().resolve()
RAW_DIR = Path(data_dir).resolve() if data_dir is not None else PROJECT_DIR / 'data'
OUT = Path(output_dir).resolve() if output_dir is not None else PROJECT_DIR / 'cleaning_outputs'
if RAW_DIR == OUT or RAW_DIR in OUT.parents:
    raise ValueError('Output directory must be outside the original data directory.')
for table in TABLES:
    if not (RAW_DIR / f'{table}.parquet').is_file():
        raise FileNotFoundError(f'Missing {RAW_DIR / (table + ".parquet")}. Open the notebook from the project root.')
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 30)
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def save(df, name):
    df.to_csv(OUT / (name + '.csv'), index=False, encoding='utf-8-sig', lineterminator='\n')
    return df
print('Output folder:', OUT)

Output folder: C:\Users\yexun\OneDrive\Desktop\Python-Group-Project\cleaning_outputs


## 1. State the project vision and question
**Vision:** Build a financial intelligence dashboard integrating market prices, news digests, and filing excerpts for the Magnificent Seven.

**Business question:** How do observed daily stock returns differ between company-days with and without recorded filing events among the Magnificent Seven?

**Metrics:** Adjusted-close returns between consecutive observed trading rows, mean return with its nonmissing denominator, trading volume, and recorded digest/filing availability. Comparisons describe associations; they do not establish causality. ADBE is excluded solely because it is outside the project scope.

## 2. Define one row
| Table | Row meaning | Candidate logical key |
|---|---|---|
| prices | One company's OHLCV observation on one trading date | symbol + date |
| news | One daily news digest record for one company, not one original article | symbol + date; investigate repeated keys |
| filings | A filing excerpt for one company, date, and document type | symbol + date + document_type; repeated keys may represent different records |

`id` is a source identifier within each table; it is not assumed unique across tables. `_source_row` records the zero-based original row position for lineage.
All conflicting price-key records are quarantined rather than choosing an arbitrary value. Nonidentical news/filing records with the same logical key are retained and aggregated before merging.

## 3. Preserve the source
Source dataset: [TheFinAI/Herculean](https://huggingface.co/datasets/TheFinAI/Herculean). Inputs are the three existing Parquet files in `data/`.
An upstream revision is not inferred from filenames. The run manifest records the local filename, size, SHA256, observed row count, and inspection timestamp. Filesystem modification times are not claimed as download dates. Add the actual download date/revision to your report if known.
Cleaning operates on copies; before/after hashes verify that local source bytes remain unchanged.

News availability is descriptive context: if every price day has a digest, there is no no-news comparison group. The main comparison therefore uses filing availability.

In [3]:
manifest_path = OUT / 'local_source_manifest.json'
previous_manifest = json.loads(manifest_path.read_text('utf-8')) if manifest_path.exists() else {}
manifest, raw = {}, {}
for table in TABLES:
    path = RAW_DIR / f'{table}.parquet'
    digest = sha(path)
    if table in previous_manifest:
        assert previous_manifest[table]['sha256'] == digest, f'Input snapshot changed: {table}. Preserve prior results and choose a new output directory.'
    raw[table] = pd.read_parquet(path)
    manifest[table] = dict(dataset=REPO, filename=path.name, sha256=digest,
        bytes=path.stat().st_size, rows=len(raw[table]), version=SOURCE_VERSION,
        first_inspected_utc=previous_manifest.get(table, {}).get('first_inspected_utc', datetime.now(timezone.utc).isoformat()))
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
display(pd.DataFrame(manifest).T)
for table, frame in raw.items():
    print(table, frame.shape)
    display(frame.head(2))
    display(pd.DataFrame({'dtype':frame.dtypes.astype(str), 'missing':frame.isna().sum(), 'distinct':frame.nunique(dropna=False)}))

,dataset,filename,sha256,bytes,rows,version,first_inspected_utc
prices,TheFinAI/Herculean,prices.parquet,736c124db22e976dbf4ec1dd65e5a983497e5890e7999f7925b343dee5c965d5,67349,2656,Local snapshot; identify exact files by SHA256. Upstream revision not assumed.,2026-10-04T20:47:28.439297+00:00
news,TheFinAI/Herculean,news.parquet,1fd109796eada02af37a15178cc150ec252e7c02bc440c3f76641eb8fab6da25,5216446,3888,Local snapshot; identify exact files by SHA256. Upstream revision not assumed.,2026-10-04T20:47:28.532238+00:00
filings,TheFinAI/Herculean,filings.parquet,822a854865651e042c9d64288785efa0f2c0fecdec2a16016ff3c2e7a5601299,600603,73,Local snapshot; identify exact files by SHA256. Upstream revision not assumed.,2026-10-04T20:47:28.544044+00:00


prices (2656, 9)


,id,symbol,date,open,high,low,close,adj_close,volume
0,1,AAPL,2024-12-02,237.270004,240.789993,237.160004,239.589996,238.288300,48137100
1,2,AAPL,2024-12-03,239.809998,242.759995,238.899994,242.649994,241.331665,38861000


,dtype,missing,distinct
id,int32,0,2656
symbol,object,0,8
date,object,0,332
open,float64,0,2556
high,float64,0,2562
low,float64,0,2562
close,float64,0,2579
adj_close,float64,0,2626
volume,int64,0,2654


news (3888, 4)


,id,symbol,date,highlights
0,1,AAPL,2024-12-01,"Apple’s December 1, 2024 coverage centers on its fiscal first quarter 2024 results, with Shackne..."
1,2,AAPL,2024-12-02,"The December 2, 2024 Apple coverage portrays a continuation of a modest upward trajectory over a..."


,dtype,missing,distinct
id,int32,0,3888
symbol,object,0,8
date,object,0,486
highlights,object,0,3888


filings (73, 6)


,id,symbol,date,document_type,mda_content,risk_content
0,1,AAPL,2024-02-01,10-Q,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,"Item 1A. Risk Factors \n\nThe Company&#8217;s business, reputation, results of operations, fina..."
1,2,AAPL,2024-05-02,10-Q,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,"Item 1A. Risk Factors \n\nThe Company&#8217;s business, reputation, results of operations, fina..."


,dtype,missing,distinct
id,int32,0,73
symbol,object,0,8
date,object,0,57
document_type,object,0,2
mda_content,object,0,73
risk_content,object,0,66


## 4–7. Declare cleaning rules before applying them
The following decision log specifies each rule, reason, and check before cleaning. A zero problem count is valid evidence; no artificial problems are introduced into the source.

- Parse dates to timezone-naive daily values. Quarantine invalid dates or missing join keys. Failed numeric conversions remain missing; do not impute means.
- Preserve identifiers as strings, including any leading zeros present in the source. Trim and uppercase symbol/form codes. For long text, trim boundaries and normalize line endings only; preserve case, HTML entities, and internal paragraph structure.
- Remove exact duplicate source records, excluding newly added lineage columns from the comparison. Report nonidentical repeated IDs separately and retain them for investigation.
- Do not impute prices, volume, news, or filing text. Whitespace-only strings become missing; the literal text NA is not automatically treated as missing.
- Replace impossible numeric values with missing and flag the record. Flag OHLC inconsistencies and unusual returns without guessing corrected values. Preserve evidence of changes.

In [4]:
rules = pd.DataFrame([
 ('scope','Keep MAG7 only','Project scope; ADBE is not bad data','row reconciliation + excluded records'),
 ('types','Parse dates/numbers; retain IDs as strings','Analysis needs comparable types','conversion failures + dtypes'),
 ('text','Trim boundaries; normalize line endings; uppercase codes','Remove formatting variation; preserve prose','text_changes.csv'),
 ('exact_duplicates','Drop repeated identical original rows, retain first','Avoid double counting identical source records','duplicates.csv + excluded_*.csv'),
 ('keys','Quarantine missing symbol/date and conflicting price keys','Cannot safely identify/join these rows','quarantine + logical duplicate counts'),
 ('missing','Keep unknown values missing; no imputation','Avoid fabricating evidence','missing_before_after.csv'),
 ('numeric_domain','Nonpositive price, negative/fractional volume, nonfinite -> missing','Invalid domain for these metrics','suspicious.csv + numeric_changes.csv'),
 ('unusual','Flag OHLC inconsistency, zero volume, large returns, unexpected forms','Unusual does not prove error','suspicious.csv'),
 ('merge','Aggregate events by symbol/date before left join','Prevent many-to-many multiplication','merge_audit.csv'),
 ('return','Adjusted price ratio minus one, no forward fill','Use only valid consecutive observed rows','independent Decimal calculation + counts'),
], columns=['rule','action','reason','check'])
display(rules)
save(rules, 'cleaning_log')

PRICE_COLS = ['open','high','low','close','adj_close']
NUMERIC = PRICE_COLS + ['volume']
KEYS = {'prices':['symbol','date'], 'news':['symbol','date'],
        'filings':['symbol','date','document_type']}
required = {'prices':['id','symbol','date',*NUMERIC],
            'news':['id','symbol','date','highlights'],
            'filings':['id','symbol','date','document_type','mda_content','risk_content']}
for table in TABLES:
    assert set(required[table]) <= set(raw[table]), f'Unexpected schema: {table}'

,rule,action,reason,check
0,scope,Keep MAG7 only,Project scope; ADBE is not bad data,row reconciliation + excluded records
1,types,Parse dates/numbers; retain IDs as strings,Analysis needs comparable types,conversion failures + dtypes
2,text,Trim boundaries; normalize line endings; uppercase codes,Remove formatting variation; preserve prose,text_changes.csv
3,exact_duplicates,"Drop repeated identical original rows, retain first",Avoid double counting identical source records,duplicates.csv + excluded_*.csv
4,keys,Quarantine missing symbol/date and conflicting price keys,Cannot safely identify/join these rows,quarantine + logical duplicate counts
5,missing,Keep unknown values missing; no imputation,Avoid fabricating evidence,missing_before_after.csv
6,numeric_domain,"Nonpositive price, negative/fractional volume, nonfinite -> missing",Invalid domain for these metrics,suspicious.csv + numeric_changes.csv
7,unusual,"Flag OHLC inconsistency, zero volume, large returns, unexpected forms",Unusual does not prove error,suspicious.csv
8,merge,Aggregate events by symbol/date before left join,Prevent many-to-many multiplication,merge_audit.csv
9,return,"Adjusted price ratio minus one, no forward fill",Use only valid consecutive observed rows,independent Decimal calculation + counts


### 12. Predeclare five real-record checks before cleaning
Checks cover an excluded ADBE row, the undefined return of the first price observation, a retained date, news-text preservation, and actual filing-text normalization. Expected results are saved before cleaning; they are not copied from the cleaned output.
The missing first return and scope exclusion are edge cases. Filing text demonstrates a real formatting change when present; unchanged text is reported honestly. A separate artificial missing-price example is labeled as synthetic verification, not a source record.

In [5]:
def text_expected(value):
    if pd.isna(value): return None
    result = str(value).replace('\r\n','\n').replace('\r','\n').strip()
    return result or None
def first_row(table, predicate):
    matches = [i for i, row in raw[table].iterrows() if predicate(row)]
    if not matches: raise ValueError(f'No record for planned check: {table}')
    return int(matches[0])
p_keep = int(raw['prices'].loc[raw['prices'].symbol.astype(str).str.strip().str.upper().isin(MAG7)].sort_values(['symbol','date']).index[0])
p_drop = first_row('prices', lambda r: str(r.symbol).strip().upper() == 'ADBE')
n_ids = [i for i,r in raw['news'].iterrows() if str(r.symbol).strip().upper() in MAG7]
n_change = [i for i in n_ids if text_expected(raw['news'].loc[i,'highlights']) != raw['news'].loc[i,'highlights']]
n_pick = int((n_change or n_ids)[0])
f_pick = first_row('filings', lambda r: str(r.symbol).strip().upper() in MAG7)
plans = [
 dict(table='prices', source_row=p_drop, field='row_presence', original='ADBE', expected='excluded:scope', reason='ADBE outside MAG7'),
 dict(table='prices', source_row=p_keep, field='daily_return', original=str(raw['prices'].loc[p_keep,'adj_close']), expected=None, reason='First observed price for this symbol has no prior denominator; return must be missing'),
 dict(table='prices', source_row=p_keep, field='date', original=str(raw['prices'].loc[p_keep,'date']), expected=str(raw['prices'].loc[p_keep,'date'])[:10], reason='Preserve calendar date'),
 dict(table='news', source_row=n_pick, field='highlights', original=raw['news'].loc[n_pick,'highlights'], expected=text_expected(raw['news'].loc[n_pick,'highlights']), reason='Boundary/newline normalization; preserve internal text'),
 dict(table='filings', source_row=f_pick, field='mda_content', original=raw['filings'].loc[f_pick,'mda_content'], expected=text_expected(raw['filings'].loc[f_pick,'mda_content']), reason='Normalize actual filing text boundaries and line endings without changing internal content'),
]
# Make the five checks distinct source records, including a second retained price row.
p_second = first_row('prices', lambda r: str(r.symbol).strip().upper() in MAG7 and r.name != p_keep)
plans[2].update(source_row=p_second, original=str(raw['prices'].loc[p_second,'date']), expected=str(raw['prices'].loc[p_second,'date'])[:10])
save(pd.DataFrame(plans), 'record_check_plan')
display(pd.DataFrame(plans))

,table,source_row,field,original,expected,reason
0,prices,332,row_presence,ADBE,excluded:scope,ADBE outside MAG7
1,prices,0,daily_return,238.28829956054688,None,First observed price for this symbol has no prior denominator; return must be missing
2,prices,1,date,2024-12-03,2024-12-03,Preserve calendar date
3,news,0,highlights,"Apple’s December 1, 2024 coverage centers on its fiscal first quarter 2024 results, with Shackne...","Apple’s December 1, 2024 coverage centers on its fiscal first quarter 2024 results, with Shackne...",Boundary/newline normalization; preserve internal text
4,filings,0,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...,Normalize actual filing text boundaries and line endings without changing internal content


In [6]:
def clean_all(source):
    cleaned, excluded, audit = {}, {}, {'types':[], 'text':[], 'duplicates':[],
        'missing':[], 'rows':[], 'numeric_changes':[]}
    for table, original in source.items():
        d = original.copy(deep=True)
        d['_source_row'] = np.arange(len(d))
        reason = pd.Series('', index=d.index, dtype='string')
        exact = original.duplicated(keep='first')
        reason.loc[exact] = 'exact_duplicate'
        for col in original.columns:
            before = d[col].copy()
            if col == 'date':
                d[col] = pd.to_datetime(before, errors='coerce', utc=True).dt.tz_convert(None).dt.normalize()
            elif table == 'prices' and col in NUMERIC:
                d[col] = pd.to_numeric(before, errors='coerce').astype('float64')
            else:
                d[col] = before.astype('string').str.replace('\r\n','\n',regex=False).str.replace('\r','\n',regex=False).str.strip().replace('',pd.NA)
                if col in ['symbol','document_type']: d[col] = d[col].str.upper()
                changed = (before.astype('string').fillna('<NULL>') != d[col].fillna('<NULL>'))
                for idx in d.index[changed]:
                    audit['text'].append(dict(table=table, source_row=int(idx), column=col,
                        before=before.loc[idx], after=d.loc[idx,col]))
            audit['types'].append(dict(table=table,column=col,before=str(before.dtype),after=str(d[col].dtype),
                conversion_failures=int((before.notna() & d[col].isna()).sum()) if col=='date' or (table=='prices' and col in NUMERIC) else 0))
        invalid_key = d['symbol'].isna() | d['date'].isna()
        reason.loc[(reason=='') & invalid_key] = 'invalid_key'
        reason.loc[(reason=='') & ~d['symbol'].isin(MAG7)] = 'scope'
        active = d.loc[reason=='']
        repeated_id = active['id'].notna() & active['id'].duplicated(keep=False)
        logical = active.duplicated(KEYS[table], keep=False)
        audit['duplicates'].append(dict(table=table, exact_extra_rows=int(exact.sum()),
            raw_repeated_id_rows=int(original['id'].duplicated(keep=False).sum()),
            scoped_repeated_id_rows=int(repeated_id.sum()), scoped_logical_duplicate_rows=int(logical.sum()),
            scoped_logical_duplicate_groups=int(active.loc[logical,KEYS[table]].drop_duplicates().shape[0])))
        if table=='prices': reason.loc[active.index[logical]] = 'conflicting_price_key'
        d['_disposition'] = reason
        excluded[table] = d.loc[reason!=''].copy()
        d = d.loc[reason==''].drop(columns='_disposition').copy()
        if table=='prices':
            d['flag_invalid_numeric'] = False
            for col in NUMERIC:
                bad = d[col].notna() & (~np.isfinite(d[col]) | ((d[col]<=0) if col in PRICE_COLS else ((d[col]<0) | (d[col]%1!=0))))
                for idx in d.index[bad]:
                    audit['numeric_changes'].append(dict(table=table, source_row=int(d.loc[idx,'_source_row']),column=col,before=d.loc[idx,col],after=np.nan))
                d.loc[bad,'flag_invalid_numeric'] = True
                d.loc[bad,col] = np.nan
            d['flag_ohlc'] = ((d.high < d.low) | (d.high < d.open) | (d.high < d.close) | (d.low > d.open) | (d.low > d.close))
            d['flag_zero_volume'] = d.volume.eq(0)
        d = d.sort_values(['symbol','date','_source_row'],kind='stable').reset_index(drop=True)
        for col in original.columns:
            audit['missing'].append(dict(table=table,column=col,before=int(original[col].isna().sum()),
                after=int(d[col].isna().sum()),before_rows=len(original),after_rows=len(d)))
        counts = reason[reason!=''].value_counts().to_dict()
        audit['rows'].append(dict(table=table,start=len(original),exact_duplicate=counts.get('exact_duplicate',0),
            invalid_key=counts.get('invalid_key',0),scope=counts.get('scope',0),
            conflicting_price_key=counts.get('conflicting_price_key',0),final=len(d)))
        cleaned[table] = d
    return cleaned, excluded, audit

cleaned, excluded, audit = clean_all(raw)
for key, filename in [('types','type_checks'),('text','text_changes'),('duplicates','duplicates'),
                      ('missing','missing_before_after'),('rows','row_reconciliation'),('numeric_changes','numeric_changes')]:
    evidence = pd.DataFrame(audit[key])
    if evidence.empty: evidence = pd.DataFrame(columns=['table','source_row','column','before','after'])
    save(evidence, filename)
    print(filename, 'records:', len(evidence))
    display(evidence.head(30))
for table in TABLES:
    save(excluded[table], f'excluded_{table}')
    if table!='prices':
        save(cleaned[table].loc[cleaned[table].duplicated(KEYS[table],keep=False)], f'logical_duplicates_{table}')

type_checks records: 19


,table,column,before,after,conversion_failures
0,prices,id,int32,string,0
1,prices,symbol,object,string,0
2,prices,date,object,datetime64[ns],0
3,prices,open,float64,float64,0
4,prices,high,float64,float64,0
5,prices,low,float64,float64,0
6,prices,close,float64,float64,0
7,prices,adj_close,float64,float64,0
8,prices,volume,int64,float64,0
9,news,id,int32,string,0


text_changes records: 146


,table,source_row,column,before,after
0,filings,0,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
1,filings,1,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
2,filings,2,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
3,filings,3,mda_content,Item 7. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 7. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
4,filings,4,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
5,filings,5,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
6,filings,6,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
7,filings,7,mda_content,Item 7. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 7. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
8,filings,8,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
9,filings,9,mda_content,ITEM 7. MANAGEMENT&#8217;S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERAT...,ITEM 7. MANAGEMENT&#8217;S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATI...


duplicates records: 3


,table,exact_extra_rows,raw_repeated_id_rows,scoped_repeated_id_rows,scoped_logical_duplicate_rows,scoped_logical_duplicate_groups
0,prices,0,0,0,0,0
1,news,0,0,0,0,0
2,filings,0,0,0,0,0


... (2 outputs left out)

row_reconciliation records: 3


,table,start,exact_duplicate,invalid_key,scope,conflicting_price_key,final
0,prices,2656,0,0,332,0,2324
1,news,3888,0,0,486,0,3402
2,filings,73,0,0,10,0,63


numeric_changes records: 0


,table,source_row,column,before,after


## 8. Check suspicious values
Full-data checks cover invalid dates, future dates relative to the execution date in UTC, impossible numeric values, OHLC ordering, zero volume, absolute returns above 30%, unexpected filing forms, and missing text. Observed date ranges are calculated separately for each table rather than assumed from the dataset card.
Gaps longer than four calendar days are review flags and may include holidays. This notebook does not use an exchange trading calendar.

In [7]:
prices, news, filings = (cleaned[t] for t in TABLES)
prices['previous_adj_close'] = prices.groupby('symbol',dropna=False)['adj_close'].shift(1)
prices['daily_return'] = prices['adj_close'] / prices['previous_adj_close'] - 1
prices['gap_days'] = prices.groupby('symbol')['date'].diff().dt.days
prices['flag_large_return'] = prices.daily_return.abs().gt(0.30)
prices['flag_date_gap'] = prices.gap_days.gt(4)
today = pd.Timestamp.now(tz='UTC').tz_localize(None).normalize()
suspicious = []
for table, d in cleaned.items():
    d['flag_future_date'] = d.date.gt(today)
    if table=='news': d['flag_missing_text'] = d.highlights.isna()
    if table=='filings':
        d['flag_form'] = ~d.document_type.isin(['10-K','10-Q'])
        d['flag_missing_text'] = d[['mda_content','risk_content']].isna().any(axis=1)
    for flag in [c for c in d if c.startswith('flag_')]:
        suspicious.append(dict(table=table,check=flag,count=int(d[flag].sum()),action='keep flagged; invalid numeric cells set missing'))
    display(d.groupby('symbol').agg(rows=('date','size'),first_date=('date','min'),last_date=('date','max')))
    flag_cols = [c for c in d if c.startswith('flag_')]
    save(d.loc[d[flag_cols].any(axis=1)], f'flagged_{table}')
save(pd.DataFrame(suspicious), 'suspicious')
display(pd.DataFrame(suspicious))

,rows,first_date,last_date
symbol,,,
AAPL,332,2024-12-02,2026-03-31
AMZN,332,2024-12-02,2026-03-31
GOOGL,332,2024-12-02,2026-03-31
META,332,2024-12-02,2026-03-31
MSFT,332,2024-12-02,2026-03-31
NVDA,332,2024-12-02,2026-03-31
TSLA,332,2024-12-02,2026-03-31


,rows,first_date,last_date
symbol,,,
AAPL,486,2024-12-01,2026-03-31
AMZN,486,2024-12-01,2026-03-31
GOOGL,486,2024-12-01,2026-03-31
META,486,2024-12-01,2026-03-31
MSFT,486,2024-12-01,2026-03-31
NVDA,486,2024-12-01,2026-03-31
TSLA,486,2024-12-01,2026-03-31


,rows,first_date,last_date
symbol,,,
AAPL,9,2024-02-01,2026-01-30
AMZN,9,2024-02-01,2026-02-05
GOOGL,9,2024-01-30,2026-02-04
META,9,2024-02-01,2026-01-28
MSFT,9,2024-01-30,2026-01-28
NVDA,9,2024-02-21,2026-02-25
TSLA,9,2024-01-26,2026-01-28


,table,check,count,action
0,prices,flag_invalid_numeric,0,keep flagged; invalid numeric cells set missing
1,prices,flag_ohlc,0,keep flagged; invalid numeric cells set missing
2,prices,flag_zero_volume,0,keep flagged; invalid numeric cells set missing
3,prices,flag_large_return,0,keep flagged; invalid numeric cells set missing
4,prices,flag_date_gap,0,keep flagged; invalid numeric cells set missing
5,prices,flag_future_date,0,keep flagged; invalid numeric cells set missing
6,news,flag_future_date,0,keep flagged; invalid numeric cells set missing
7,news,flag_missing_text,0,keep flagged; invalid numeric cells set missing
8,filings,flag_future_date,0,keep flagged; invalid numeric cells set missing
9,filings,flag_form,0,keep flagged; invalid numeric cells set missing


## 9. Verify merges
Join on `symbol,date`. Aggregate news and filings to one row per company-day, then perform validated **one-to-one left joins** to unique price keys.
`news_record_count` counts digest records, not original articles. Fill unmatched event counts with zero, meaning no record in this dataset; do not fabricate text.
Report and export unmatched keys from both sides. Weekend news and filings outside the price window are not reassigned to trading days. Daily dates cannot distinguish premarket from after-hours information or establish a trading strategy without look-ahead bias.

In [8]:
news_daily = news.groupby(['symbol','date'],as_index=False).agg(
    news_record_count=('_source_row','size'),news_nonmissing_text=('highlights','count'))
filings_daily = filings.groupby(['symbol','date'],as_index=False).agg(filing_record_count=('_source_row','size'))
panel = prices.copy()
merge_rows = []
for label, right in [('news',news_daily),('filings',filings_daily)]:
    keys = ['symbol','date']
    coverage = prices[keys].merge(right[keys],on=keys,how='outer',indicator=True,validate='one_to_one')
    save(coverage.loc[coverage['_merge']!='both'], f'unmatched_{label}_keys')
    before = len(panel)
    panel = panel.merge(right,on=keys,how='left',validate='one_to_one',indicator=True)
    merge_rows.append(dict(table=label,keys='symbol,date',relationship='one_to_one after aggregation',
        before=before,after=len(panel),extra_rows=len(panel)-before,
        price_keys_without_event=int(coverage['_merge'].eq('left_only').sum()),
        event_keys_without_price=int(coverage['_merge'].eq('right_only').sum()),
        matched_keys=int(coverage['_merge'].eq('both').sum())))
    panel = panel.drop(columns='_merge')
for c in ['news_record_count','news_nonmissing_text','filing_record_count']:
    panel[c] = panel[c].fillna(0).astype('int64')
assert len(panel)==len(prices) and not panel.duplicated(['symbol','date']).any()
assert np.isclose(panel.volume.sum(min_count=1),prices.volume.sum(min_count=1),equal_nan=True)
save(pd.DataFrame(merge_rows),'merge_audit')
display(pd.DataFrame(merge_rows))

,table,keys,relationship,before,after,extra_rows,price_keys_without_event,event_keys_without_price,matched_keys
0,news,"symbol,date",one_to_one after aggregation,2324,2324,0,0,1078,2324
1,filings,"symbol,date",one_to_one after aggregation,2324,2324,0,2289,28,35


## 10. Reconcile changes
Each source record has at most one exclusion reason, assigned in order: exact duplicate → invalid key → scope → conflicting price key.
Reconcile row counts and finite, parseable volume totals. Original volume equals excluded-category volume plus retained original volume. Removing invalid retained volume yields the cleaned total. Verify that merging does not change it.
Summed share volume across securities is an accounting check, not dollar value or a directly comparable measure of trading activity.

In [9]:
reconciliation = pd.DataFrame(audit['rows'])
assert (reconciliation.start == reconciliation[['exact_duplicate','invalid_key','scope','conflicting_price_key','final']].sum(axis=1)).all()
v = pd.to_numeric(raw['prices']['volume'],errors='coerce').replace([np.inf,-np.inf],np.nan)
volume_parts = []
for reason, g in excluded['prices'].groupby('_disposition'):
    volume_parts.append(dict(stage=reason, rows=len(g), volume=float(v.iloc[g._source_row].sum())))
retained_original_volume = float(v.iloc[prices._source_row].sum())
volume_parts += [dict(stage='retained_before_domain_cleaning',rows=len(prices),volume=retained_original_volume)]
assert np.isclose(v.sum(),sum(x['volume'] for x in volume_parts))
volume_parts += [dict(stage='invalid_volume_removed_from_retained',rows=int(sum(x['column']=='volume' for x in audit['numeric_changes'])),volume=retained_original_volume-float(prices.volume.sum())),
                 dict(stage='cleaned',rows=len(prices),volume=float(prices.volume.sum())),
                 dict(stage='merged',rows=len(panel),volume=float(panel.volume.sum()))]
save(pd.DataFrame(volume_parts),'volume_reconciliation')
display(reconciliation)
display(pd.DataFrame(volume_parts))

,table,start,exact_duplicate,invalid_key,scope,conflicting_price_key,final
0,prices,2656,0,0,332,0,2324
1,news,3888,0,0,486,0,3402
2,filings,73,0,0,10,0,63


,stage,rows,volume
0,scope,332,1.454422e+09
1,retained_before_domain_cleaning,2324,1.580024e+11
2,invalid_volume_removed_from_retained,0,0.000000e+00
3,cleaned,2324,1.580024e+11
4,merged,2324,1.580024e+11


## 11. Check calculations — independently validate the Codex return suggestion
Formula: `adj_close[t] / adj_close[t-1] - 1`. Returns are fractions: 0.01 means 1%. The denominator is the previous observed row for the same company; never cross company boundaries or forward-fill missing values.
The first row for each company and rows with a missing current or previous value have missing returns, not zero returns. A missing trading date can make an observed-row return span multiple days.
An independent Python/Decimal loop verifies **every return** and reports valid observation counts. Rows with missing symbol/date keys were explicitly quarantined rather than silently lost during grouping.

In [10]:
from decimal import Decimal
independent = []
for symbol, g in prices.groupby('symbol',sort=False):
    prev = None
    for row in g.itertuples():
        current = None if pd.isna(row.adj_close) else Decimal(str(row.adj_close))
        value = np.nan if prev is None or current is None else float(current / prev - Decimal(1))
        independent.append(value)
        prev = current
np.testing.assert_allclose(prices.daily_return.to_numpy(), independent, rtol=1e-10,atol=1e-12,equal_nan=True)
# Artificial verification example, not a source-data record.
toy = pd.Series([100.0,110.0,np.nan,121.0,133.1])
np.testing.assert_allclose((toy/toy.shift()-1).to_numpy(),[np.nan,.1,np.nan,np.nan,.1],equal_nan=True,atol=1e-12)
metrics = prices.groupby('symbol').agg(rows=('date','size'),valid_returns=('daily_return','count'),
    mean_return=('daily_return','mean'),valid_volume=('volume','count'),mean_volume=('volume','mean'))
metrics['volume_sum'] = prices.groupby('symbol').volume.sum(min_count=1)
metrics['missing_returns'] = metrics.rows - metrics.valid_returns
save(metrics.reset_index(),'metrics')
display(metrics)
comparison = panel.assign(has_filing=panel.filing_record_count.gt(0)).groupby(
    'has_filing',dropna=False).agg(rows=('date','size'),valid_returns=('daily_return','count'),mean_return=('daily_return','mean')).reset_index()
save(comparison,'descriptive_comparison')
display(comparison)
print('Price days with news:', int(panel.news_record_count.gt(0).sum()), 'of', len(panel))
if panel.news_record_count.gt(0).all():
    print('No no-news comparison group exists. News availability cannot explain between-group return differences here.')
print('PASS: every return independently checked; missing-value toy example passed.')

,rows,valid_returns,mean_return,valid_volume,mean_volume,volume_sum,missing_returns
symbol,,,,,,,
AAPL,332,331,0.000372,332,5.256276e+07,1.745083e+10,1
AMZN,332,331,0.000187,332,4.513671e+07,1.498539e+10,1
GOOGL,332,331,0.001776,332,3.557789e+07,1.181186e+10,1
META,332,331,0.000193,332,1.538954e+07,5.109327e+09,1
MSFT,332,331,-0.000297,332,2.475262e+07,8.217871e+09,1
NVDA,332,331,0.001127,332,2.118627e+08,7.033842e+10,1
TSLA,332,331,0.000820,332,9.062856e+07,3.008868e+10,1


,has_filing,rows,valid_returns,mean_return
0,False,2289,2282,0.000591
1,True,35,35,0.000951


Price days with news: 2324 of 2324
No no-news comparison group exists. News availability cannot explain between-group return differences here.
PASS: every return independently checked; missing-value toy example passed.


## 12. Verify individual records against the predeclared expectations
The CSV retains full text; only the on-screen display is abbreviated. A mismatch stops execution for investigation. Expected values are not rewritten to match the output.

In [11]:
checks=[]
for plan in plans:
    table, source_row, field = plan['table'],plan['source_row'],plan['field']
    found = cleaned[table].loc[cleaned[table]._source_row.eq(source_row)]
    if field=='row_presence':
        dropped = excluded[table].loc[excluded[table]._source_row.eq(source_row)]
        actual = 'retained' if len(found) else 'excluded:' + str(dropped.iloc[0]['_disposition'])
    elif found.empty: actual = '<ROW EXCLUDED>'
    else:
        value = found.iloc[0][field]
        actual = None if pd.isna(value) else (value.strftime('%Y-%m-%d') if field=='date' else str(value))
    checks.append({**plan,'actual':actual,'pass':actual==plan['expected']})
record_checks = save(pd.DataFrame(checks),'record_checks')
display(record_checks)
assert len(record_checks)==5 and record_checks['pass'].all()

,table,source_row,field,original,expected,reason,actual,pass
0,prices,332,row_presence,ADBE,excluded:scope,ADBE outside MAG7,excluded:scope,True
1,prices,0,daily_return,238.28829956054688,None,First observed price for this symbol has no prior denominator; return must be missing,None,True
2,prices,1,date,2024-12-03,2024-12-03,Preserve calendar date,2024-12-03,True
3,news,0,highlights,"Apple’s December 1, 2024 coverage centers on its fiscal first quarter 2024 results, with Shackne...","Apple’s December 1, 2024 coverage centers on its fiscal first quarter 2024 results, with Shackne...",Boundary/newline normalization; preserve internal text,"Apple’s December 1, 2024 coverage centers on its fiscal first quarter 2024 results, with Shackne...",True
4,filings,0,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...,Normalize actual filing text boundaries and line endings without changing internal content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...,True


## 13. Test reproducibility and export evidence
Write cleaned tables (CSV/Parquet), the merged panel, at most 50 sample rows, cleaning and verification evidence, and the observed package versions. Do not rewrite the project's README or dependency files.

Sample priority: predeclared retained checks, flagged records, missing source fields, company coverage/first observed rows, then stable-order context. This purposive sample is not representative for statistical estimation.

Reload local originals and rerun the cleaning core; verify frame equality and unchanged source hashes. The first complete run establishes a data/evidence hash baseline. **Restart the kernel and Run All again** to check a second full execution against it. The script can likewise be run twice. Download/inspection metadata and checklist status are excluded from byte comparisons.
If an intentional source or code change alters outputs, preserve the earlier results and use a new output folder rather than silently replacing a failing baseline.

In [12]:
fresh = {t:pd.read_parquet(RAW_DIR/f'{t}.parquet') for t in TABLES}
recleaned, reexcluded, reaudit = clean_all(fresh)
for table in TABLES:
    base_cols = recleaned[table].columns
    pd.testing.assert_frame_equal(recleaned[table],cleaned[table][base_cols])
    pd.testing.assert_frame_equal(reexcluded[table],excluded[table])
    assert sha(RAW_DIR/f'{table}.parquet')==manifest[table]['sha256']
    save(cleaned[table],f'cleaned_{table}')
    cleaned[table].to_parquet(OUT/f'cleaned_{table}.parquet',index=False)
save(panel,'analysis_panel')
samples=[]
for table, quota in [('prices',25),('news',15),('filings',10)]:
    d=cleaned[table]
    priorities={}
    def add(ids, reason):
        for i in ids: priorities.setdefault(int(i),reason)
    add([p['source_row'] for p in plans if p['table']==table and p['field']!='row_presence'],'predeclared record check')
    flags=[c for c in d if c.startswith('flag_')]
    add(d.loc[d[flags].any(axis=1),'_source_row'],'flagged difficult case')
    add(d.loc[d[required[table]].isna().any(axis=1),'_source_row'],'missing field')
    add(d.groupby('symbol',sort=True).head(1)._source_row,'symbol coverage / first observed row')
    add(d._source_row,'stable-order context')
    ids=[i for i in priorities if i in set(d._source_row)][:quota]
    chosen=d.set_index('_source_row').loc[ids].reset_index()
    chosen.insert(0,'table',table)
    chosen.insert(1,'sample_reason',[priorities[i] for i in ids])
    samples.append(chosen)
sample=pd.concat(samples,ignore_index=True,sort=False)
assert len(sample)<=50
save(sample,'cleaned_sample_max50')
versions={name:importlib.metadata.version(name) for name in ['pandas','numpy','pyarrow']}
(OUT/'environment.json').write_text(json.dumps({'python':platform.python_version(),'packages':versions,'source_version':SOURCE_VERSION},indent=2),encoding='utf-8')
# Checklist changes after the second run; exclude it from deterministic data hashes.
data_files=sorted(p for p in OUT.iterdir() if p.suffix in ['.csv','.parquet'] and p.name!='checklist_14.csv')
current_hashes={p.name:sha(p) for p in data_files}
hash_path=OUT/'output_hashes.json'
had_baseline=hash_path.exists()
if had_baseline:
    baseline=json.loads(hash_path.read_text('utf-8'))
    assert baseline==current_hashes, 'Output mismatch: investigate code/data/dependency changes; preserve old baseline.'
else:
    hash_path.write_text(json.dumps(current_hashes,indent=2),encoding='utf-8')
print('PASS: outputs match previous complete run.' if had_baseline else 'Baseline saved. Restart Kernel and Run All to confirm reproducibility.')
print('Sample rows:',len(sample))
display(sample[['table','sample_reason','symbol','date','_source_row']])

Baseline saved. Restart Kernel and Run All to confirm reproducibility.
Sample rows: 50


,table,sample_reason,symbol,date,_source_row
0,prices,predeclared record check,AAPL,2024-12-02,0
1,prices,predeclared record check,AAPL,2024-12-03,1
2,prices,symbol coverage / first observed row,AMZN,2024-12-02,664
3,prices,symbol coverage / first observed row,GOOGL,2024-12-02,996
4,prices,symbol coverage / first observed row,META,2024-12-02,1328
...,...,...,...,...,...
45,filings,symbol coverage / first observed row,NVDA,2024-02-21,55
46,filings,symbol coverage / first observed row,TSLA,2024-01-26,64
47,filings,stable-order context,AAPL,2024-05-02,1
48,filings,stable-order context,AAPL,2024-08-01,2


## 14. Document remaining limitations
1. Seven large technology companies and a limited observation window do not represent the whole market. Coverage can differ across tables.
2. News consists of model-summarized daily highlights, not original articles. Summaries may omit information; record counts are not article counts. Filing text covers selected sections only.
3. Reliable publication times and exchange time zones are unavailable. Same-day comparisons are descriptive, and weekend/out-of-window events are not mapped to price dates.
4. Upstream adjusted-price methodology is not independently verified here. Any unusual prices, OHLC conflicts, or long gaps require source and trading-calendar review. Flagged observations can affect means; if flags occur, evaluate sensitivity to excluding them.
5. Nonidentical repeated IDs or event keys could reflect legitimate records or upstream duplication. Retention affects event counts; additional source identifiers are needed to resolve ambiguity.
6. Missing or quarantined records alter coverage. Means use nonmissing returns as their denominator and may have selection bias; more complete sources are needed to evaluate the missingness mechanism.

**Codex suggestion validation:** The proposed return calculation was checked by explicitly prohibiting missing-value filling, comparing all returns against an independent Decimal implementation, and testing a small missing-value example. Explain the actual evidence and limitations in your individual PDF.

### Before submission
Review all evidence, restart and run all cells again, and save this notebook. Update the existing project README and your individual PDF with your name, branch name, branch URL, and submitted commit ID. The script does not publish, commit, or modify these documents. Include the generated sample and evidence as required by the assignment.

In [13]:
titles=['State the project vision and question','Define one row','Preserve the source','Inspect columns and types',
        'Standardize text','Investigate duplicates','Handle missing data','Check suspicious values','Verify merges',
        'Reconcile changes','Check calculations','Verify individual records','Test reproducibility','Document remaining limitations']
refs=['Section 1','Section 2; duplicates.csv','local_source_manifest.json; raw file hashes','type_checks.csv',
      'text_changes.csv (zero changes is valid)','duplicates.csv; excluded_*.csv; logical_duplicates_*.csv',
      'missing_before_after.csv','suspicious.csv; flagged_*.csv','merge_audit.csv; unmatched_*_keys.csv',
      'row_reconciliation.csv; volume_reconciliation.csv','metrics.csv; Decimal all-row assertion',
      'record_check_plan.csv; record_checks.csv','output_hashes.json; Restart & Run All comparison','Section 14']
statuses=['Done']*14
notes=['']*14
if not had_baseline:
    statuses[12]='Unresolved'; notes[12]='Baseline only; restart kernel and run all again.'
dup=pd.DataFrame(audit['duplicates'])
if dup.scoped_repeated_id_rows.sum() or dup.scoped_logical_duplicate_rows.sum():
    statuses[5]='Unresolved'; notes[5]='See duplicate evidence; conflicts require source confirmation; event counts may be inflated.'
if sum(x['count'] for x in suspicious):
    statuses[7]='Unresolved'; notes[7]='Flagged records retained; inspect flagged files; may affect returns or event coverage.'
checklist=pd.DataFrame({'item':range(1,15),'requirement':titles,'status':statuses,'evidence':refs,'explanation':notes})
save(checklist,'checklist_14')
display(checklist)
print('Saved cleaned data and verification evidence to', OUT)
print('Complete your individual PDF and project README with your name and GitHub branch details.')

,item,requirement,status,evidence,explanation
0,1,State the project vision and question,Done,Section 1,
1,2,Define one row,Done,Section 2; duplicates.csv,
2,3,Preserve the source,Done,local_source_manifest.json; raw file hashes,
3,4,Inspect columns and types,Done,type_checks.csv,
4,5,Standardize text,Done,text_changes.csv (zero changes is valid),
5,6,Investigate duplicates,Done,duplicates.csv; excluded_*.csv; logical_duplicates_*.csv,
6,7,Handle missing data,Done,missing_before_after.csv,
7,8,Check suspicious values,Done,suspicious.csv; flagged_*.csv,
8,9,Verify merges,Done,merge_audit.csv; unmatched_*_keys.csv,
9,10,Reconcile changes,Done,row_reconciliation.csv; volume_reconciliation.csv,


Saved cleaned data and verification evidence to C:\Users\yexun\OneDrive\Desktop\Python-Group-Project\cleaning_outputs
Complete your individual PDF and project README with your name and GitHub branch details.
